# Convergence and validation-threshold figures

Requires saved training-history and validation-threshold files.

Source: `graindiscoloration (2).ipynb`, zero-based cell 65.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
# ============================================================
# PUBLICATION-READY CONVERGENCE + THRESHOLD FIGURES
# Seed 42 | Six-channel model | Validation set | No TTA
# ============================================================

from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, FileLink

# ------------------------------------------------------------
# 1. Locate source files automatically
# ------------------------------------------------------------
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/gd_final_curve_figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def locate_file(filename):
    candidates = list(INPUT_ROOT.rglob(filename))

    if not candidates:
        raise FileNotFoundError(
            f"{filename} পাওয়া যায়নি। Dataset Input-এ যুক্ত আছে কিনা দেখুন।"
        )

    # Prefer the correct seed-42, without-NDRE result folder
    preferred = [
        p for p in candidates
        if "gd-v3-1-without-ndre-seed42-results" in str(p).lower()
    ]

    selected = preferred[0] if preferred else candidates[0]
    print(f"Using: {selected}")
    return selected


history_path = locate_file("training_history.csv")
threshold_path = locate_file("validation_threshold_search_no_tta.csv")

history = pd.read_csv(history_path)
threshold_df = pd.read_csv(threshold_path)

# Standardise column names
history.columns = [
    c.strip().lower().replace(" ", "_")
    for c in history.columns
]

threshold_df.columns = [
    c.strip().lower().replace(" ", "_")
    for c in threshold_df.columns
]

required_history = [
    "epoch",
    "train_main_loss",
    "val_main_loss",
    "val_dice",
    "val_iou"
]

required_threshold = [
    "threshold",
    "dice",
    "iou",
    "precision",
    "recall"
]

for column in required_history:
    if column not in history.columns:
        raise ValueError(
            f"Missing history column: {column}\n"
            f"Available columns: {history.columns.tolist()}"
        )

for column in required_threshold:
    if column not in threshold_df.columns:
        raise ValueError(
            f"Missing threshold column: {column}\n"
            f"Available columns: {threshold_df.columns.tolist()}"
        )

history = history.sort_values("epoch").reset_index(drop=True)
threshold_df = threshold_df.sort_values("threshold").reset_index(drop=True)

# ------------------------------------------------------------
# 2. Determine selected epoch and threshold
# ------------------------------------------------------------
best_epoch_index = history["val_dice"].idxmax()
best_epoch = int(history.loc[best_epoch_index, "epoch"])
training_end = int(history["epoch"].max())

best_val_dice = float(history.loc[best_epoch_index, "val_dice"])
best_val_iou = float(history.loc[best_epoch_index, "val_iou"])

best_threshold_index = threshold_df["dice"].idxmax()
selected_threshold = float(
    threshold_df.loc[best_threshold_index, "threshold"]
)

selected_values = threshold_df.loc[best_threshold_index]

print("\nSelected checkpoint epoch:", best_epoch)
print("Training endpoint:", training_end)
print("Selected threshold:", selected_threshold)

# ------------------------------------------------------------
# 3. Publication style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 1.0,
    "lines.linewidth": 2.3,
    "savefig.dpi": 600,
    "figure.dpi": 150
})

COLORS = {
    "train": "#2878B5",
    "validation": "#D55E00",
    "dice": "#D55E00",
    "iou": "#2878B5",
    "precision": "#6F63B6",
    "recall": "#1B9E77",
    "selected": "#555555",
    "endpoint": "#9A5A00"
}

# ============================================================
# FIGURE 1: TRAINING AND VALIDATION CONVERGENCE
# ============================================================
fig, axes = plt.subplots(
    1, 2,
    figsize=(13.0, 5.0)
)

fig.subplots_adjust(
    left=0.075,
    right=0.985,
    bottom=0.14,
    top=0.80,
    wspace=0.24
)

# ------------------------------------------------------------
# Panel (a): Comparable main-output losses
# ------------------------------------------------------------
ax = axes[0]

ax.plot(
    history["epoch"],
    history["train_main_loss"],
    color=COLORS["train"],
    linestyle="-",
    marker=None,
    label="Training main loss"
)

ax.plot(
    history["epoch"],
    history["val_main_loss"],
    color=COLORS["validation"],
    linestyle="-",
    marker=None,
    label="Validation main loss"
)

ax.axvline(
    best_epoch,
    color=COLORS["selected"],
    linestyle="--",
    linewidth=1.8
)

ax.axvline(
    training_end,
    color=COLORS["endpoint"],
    linestyle=":",
    linewidth=2.0
)

ax.set_xlim(1, training_end + 2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Main segmentation loss")

ax.set_xticks([1, 10, 20, 30, 40, training_end])

ax.grid(
    True,
    linestyle="-",
    linewidth=0.6,
    alpha=0.22
)

# Panel label
ax.text(
    0.02, 0.95,
    "(a)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top"
)

# Legend outside the plotting area
ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.50, 1.10),
    ncol=2,
    frameon=False
)

# Epoch labels above the axes
ax.text(
    best_epoch,
    1.015,
    f"Selected: {best_epoch}",
    transform=ax.get_xaxis_transform(),
    ha="center",
    va="bottom",
    fontsize=9,
    color=COLORS["selected"]
)

ax.text(
    training_end,
    1.015,
    f"End: {training_end}",
    transform=ax.get_xaxis_transform(),
    ha="right",
    va="bottom",
    fontsize=9,
    color=COLORS["endpoint"]
)

# ------------------------------------------------------------
# Panel (b): Validation Dice and IoU
# ------------------------------------------------------------
ax = axes[1]

ax.plot(
    history["epoch"],
    history["val_dice"],
    color=COLORS["dice"],
    linestyle="-",
    marker=None,
    label="Validation Dice"
)

ax.plot(
    history["epoch"],
    history["val_iou"],
    color=COLORS["recall"],
    linestyle="-",
    marker=None,
    label="Validation IoU"
)

ax.axvline(
    best_epoch,
    color=COLORS["selected"],
    linestyle="--",
    linewidth=1.8
)

ax.axvline(
    training_end,
    color=COLORS["endpoint"],
    linestyle=":",
    linewidth=2.0
)

# Highlight only the selected checkpoint
ax.scatter(
    [best_epoch],
    [best_val_dice],
    s=65,
    color=COLORS["dice"],
    edgecolor="white",
    linewidth=1.2,
    zorder=5
)

ax.scatter(
    [best_epoch],
    [best_val_iou],
    s=65,
    color=COLORS["recall"],
    edgecolor="white",
    linewidth=1.2,
    zorder=5
)

# Small labels instead of large overlapping annotation box
ax.text(
    0.97, 0.06,
    f"Epoch {best_epoch}\n"
    f"Dice = {best_val_dice:.3f}\n"
    f"IoU = {best_val_iou:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=9,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="0.75",
        alpha=0.92
    )
)
ax.set_xlim(1, training_end + 2)
ax.set_ylim(0.49, 0.815)

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation score")
ax.set_xticks([1, 10, 20, 30, 40, training_end])

ax.grid(
    True,
    linestyle="-",
    linewidth=0.6,
    alpha=0.22
)

ax.text(
    0.02, 0.95,
    "(b)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top"
)

ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.50, 1.10),
    ncol=2,
    frameon=False
)

ax.text(
    best_epoch,
    1.015,
    f"Selected: {best_epoch}",
    transform=ax.get_xaxis_transform(),
    ha="center",
    va="bottom",
    fontsize=9,
    color=COLORS["selected"]
)

ax.text(
    training_end,
    1.015,
    f"End: {training_end}",
    transform=ax.get_xaxis_transform(),
    ha="right",
    va="bottom",
    fontsize=9,
    color=COLORS["endpoint"]
)

# Save convergence figure
convergence_png = OUTPUT_DIR / "figure_training_validation_convergence_final.png"
convergence_pdf = OUTPUT_DIR / "figure_training_validation_convergence_final.pdf"

fig.savefig(
    convergence_png,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    convergence_pdf,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

# ============================================================
# FIGURE 2: VALIDATION THRESHOLD SENSITIVITY
# Pure solid curves — no dense markers
# ============================================================
fig, ax = plt.subplots(figsize=(8.8, 5.6))

fig.subplots_adjust(
    left=0.13,
    right=0.97,
    bottom=0.15,
    top=0.80
)

metric_styles = {
    "dice": COLORS["dice"],
    "iou": COLORS["iou"],
    "precision": COLORS["precision"],
    "recall": COLORS["recall"]
}

metric_labels = {
    "dice": "Dice",
    "iou": "IoU",
    "precision": "Precision",
    "recall": "Recall"
}

# Solid lines only
for metric in ["dice", "iou", "precision", "recall"]:
    ax.plot(
        threshold_df["threshold"],
        threshold_df[metric],
        color=metric_styles[metric],
        linestyle="-",
        linewidth=2.4,
        marker=None,
        solid_capstyle="round",
        label=metric_labels[metric]
    )

# Selected threshold
ax.axvline(
    selected_threshold,
    color="#333333",
    linestyle="--",
    linewidth=1.8
)

# Highlight only the four selected operating-point values
for metric in ["dice", "iou", "precision", "recall"]:
    ax.scatter(
        selected_threshold,
        selected_values[metric],
        s=58,
        color=metric_styles[metric],
        edgecolor="white",
        linewidth=1.1,
        zorder=5
    )

ax.set_xlabel("Segmentation threshold")
ax.set_ylabel("Validation score")

ax.set_xlim(
    threshold_df["threshold"].min() - 0.01,
    threshold_df["threshold"].max() + 0.01
)

ax.set_ylim(0.63, 0.90)

ax.grid(
    True,
    linestyle="-",
    linewidth=0.6,
    alpha=0.22
)

# Clean legend outside the graph
ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.50, 1.10),
    ncol=4,
    frameon=False,
    columnspacing=1.5
)

# Threshold label in unused upper region
ax.text(
    selected_threshold + 0.008,
    0.896,
    f"Selected threshold = {selected_threshold:.2f}",
    ha="left",
    va="top",
    fontsize=10,
    color="#333333"
)

threshold_png = OUTPUT_DIR / "figure_validation_threshold_sensitivity_final.png"
threshold_pdf = OUTPUT_DIR / "figure_validation_threshold_sensitivity_final.pdf"

fig.savefig(
    threshold_png,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    threshold_pdf,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 4. Save numerical summary
# ------------------------------------------------------------
summary = pd.DataFrame({
    "item": [
        "Selected checkpoint epoch",
        "Training endpoint epoch",
        "Selected validation Dice",
        "Selected validation IoU",
        "Selected threshold",
        "Precision at selected threshold",
        "Recall at selected threshold"
    ],
    "value": [
        best_epoch,
        training_end,
        best_val_dice,
        best_val_iou,
        selected_threshold,
        float(selected_values["precision"]),
        float(selected_values["recall"])
    ]
})

summary_path = OUTPUT_DIR / "figure_numerical_summary.csv"
summary.to_csv(summary_path, index=False)

# Create downloadable ZIP
zip_base = "/kaggle/working/gd_final_curve_figures"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=OUTPUT_DIR
)

print("\n" + "=" * 65)
print("FIGURES COMPLETED")
print("=" * 65)
print(f"Selected epoch       : {best_epoch}")
print(f"Training endpoint    : {training_end}")
print(f"Validation Dice      : {best_val_dice:.4f}")
print(f"Validation IoU       : {best_val_iou:.4f}")
print(f"Selected threshold   : {selected_threshold:.2f}")
print(f"Precision            : {selected_values['precision']:.4f}")
print(f"Recall               : {selected_values['recall']:.4f}")
print(f"Output folder        : {OUTPUT_DIR}")

display(FileLink(zip_path))